## PROJECT DESCRIPTION

This project simulates a synthetic population of tomato plants grown across multiple locations and seasons to study how environmental and management variability affects physiological responses. A synthetic dataset is generated using biologically plausible distributions and species-response assumptions, and machine-learning models are trained to predict chlorophyll content, photosynthetic rate, stomatal conductance, and relative water content. The workflow also includes climate-stress scenario simulation to quantify expected physiological impacts under increased temperature and reduced rainfall.

#### DATASET STRUCTURE (FINAL CSV SCHEMA)

**Columns**
Identifiers
sample_id (int)
location (Loc_1, Loc_2, Loc_3, Loc_4)
season (Dry, Wet)
Environmental Variables
avg_temp_c (float)
humidity_pct (float)
rainfall_mm (float)
solar_radiation_mj (float)
soil_moisture_pct (float)
soil_ph (float)
altitude_m (float)
Management Variables
fertilizer_rate_kg_ha (float)
irrigation_frequency_per_week (int)
Growth Variables
plant_age_weeks (int)
leaf_area_cm2 (float)
flowering_stage (Vegetative, Flowering, Fruiting)
Stress Indices
drought_stress_index (0–1)
pathogen_pressure_index (0–1)
Targets (Physiology Outputs)
chlorophyll_spad
photosynthetic_rate
stomatal_conductance
relative_water_content_pct
Extra Target (Very useful)
fruit_set_rate_pct

Optional derived label:

stress_class (Low/Moderate/High)

#### METHODOLOGY (RESEARCH-FRIENDLY)
* Step 1 — Synthetic Population Generation
Generate a virtual population of tomato plants (N=6000)
Simulate realistic region/location differences in climate and soil
Apply seasonal effects (wet season increases humidity and moisture)
* Step 2 — Stress Index Engineering
Build drought index using rainfall + soil moisture
Build pathogen pressure using humidity + moisture
* Step 3 — Physiological Response Modelling

Generate target variables using plausible relationships:

Drought reduces stomatal conductance and RWC
Heat reduces chlorophyll and photosynthesis
PathogenThe  pressure reduces photosynthesis and fruit set
growth stage affects fruit set and physiology
Step 4 — Machine Learning Models

Train:

Linear Regression baseline
Random Forest
Gradient Boosting

Evaluate using MAE/RMSE/R².

Step 5 — Scenario Simulation

Simulate a climate stress scenario:

+2°C temperature increase
-30% rainfall reduction
+0.2 drought index increase

Quantify predicted impacts on physiology and fruit set.

✅ REPO STRUCTURE
synthetic-tomato-physiology-model/
│
├── data/
│   └── synthetic_tomato_population.csv
│
├── notebooks/
│   └── Tomato_Physiology_Virtual_Population_Model.ipynb
│
├── src/
│   └── synthetic_population_generator.py
│
├── outputs/
│   └── figures/
│
├── README.md
├── requirements.txt
└── LICENSE
✅ REQUIREMENTS.TXT
numpy
pandas
matplotlib
scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
np.random.seed(42)

# -----------------------------
# 1. Synthetic Population Generation
# -----------------------------

N = 6000

locations = ["Loc_1", "Loc_2", "Loc_3", "Loc_4"]
seasons = ["Dry", "Wet"]
stages = ["Vegetative", "Flowering", "Fruiting"]

location = np.random.choice(locations, size=N)
season = np.random.choice(seasons, size=N)
flowering_stage = np.random.choice(stages, size=N, p=[0.35, 0.35, 0.30])

# Base environment distributions
avg_temp_c = np.random.normal(27, 3, size=N)
humidity_pct = np.random.normal(72, 12, size=N)
rainfall_mm = np.random.normal(1050, 400, size=N)
solar_radiation_mj = np.random.normal(17, 4, size=N)
soil_moisture_pct = np.random.normal(25, 8, size=N)
soil_ph = np.random.normal(6.4, 0.5, size=N)
altitude_m = np.random.normal(350, 220, size=N)

# Location effects (simulate ecological differences)
loc_temp_effect = {"Loc_1": 1.5, "Loc_2": -1.0, "Loc_3": 0.5, "Loc_4": 0.0}
loc_rain_effect = {"Loc_1": -250, "Loc_2": 250, "Loc_3": 120, "Loc_4": 0}
loc_alt_effect = {"Loc_1": 180, "Loc_2": -80, "Loc_3": 100, "Loc_4": 0}

avg_temp_c += np.array([loc_temp_effect[l] for l in location])
rainfall_mm += np.array([loc_rain_effect[l] for l in location])
altitude_m += np.array([loc_alt_effect[l] for l in location])

# Season effects
avg_temp_c += np.where(season == "Dry", 1.3, -0.9)
humidity_pct += np.where(season == "Wet", 10, -12)
rainfall_mm += np.where(season == "Wet", 320, -280)
soil_moisture_pct += np.where(season == "Wet", 10, -8)

# Management variables
fertilizer_rate_kg_ha = np.clip(np.random.normal(140, 45, size=N), 50, 250)
irrigation_frequency_per_week = np.random.poisson(2, size=N)

# Plant variables
plant_age_weeks = np.random.randint(4, 18, size=N)

leaf_area_cm2 = np.clip(
    np.random.normal(45, 18, size=N) + plant_age_weeks * 1.4,
    10, 160
)

In [ ]:
# -----------------------------
# 2. Stress Index Engineering
# -----------------------------

# Drought stress increases when rainfall and soil moisture are low
drought_stress_index = (
    1
    - (rainfall_mm / np.max(rainfall_mm)) * 0.55
    - (soil_moisture_pct / np.max(soil_moisture_pct)) * 0.45
)

drought_stress_index = np.clip(drought_stress_index + np.random.normal(0, 0.06, size=N), 0, 1)

# Pathogen pressure increases with humidity and moisture
pathogen_pressure_index = (
    (humidity_pct / np.max(humidity_pct)) * 0.65
    + (soil_moisture_pct / np.max(soil_moisture_pct)) * 0.35
)

pathogen_pressure_index = np.clip(pathogen_pressure_index + np.random.normal(0, 0.07, size=N), 0, 1)

# Growth stage numeric effect
stage_effect = np.where(flowering_stage == "Vegetative", 0.9,
                np.where(flowering_stage == "Flowering", 1.0, 1.1))

In [ ]:
# -----------------------------
# 3. Physiological Targets (Biologically plausible assumptions)
# -----------------------------

# Chlorophyll SPAD decreases with heat, drought, and pathogen pressure
chlorophyll_spad = (
    48
    - (avg_temp_c - 27) * 1.6
    - drought_stress_index * 6.5
    - pathogen_pressure_index * 5.5
    + fertilizer_rate_kg_ha * 0.02
    + np.random.normal(0, 2.5, size=N)
)

chlorophyll_spad = np.clip(chlorophyll_spad, 8, 65)

# Photosynthetic rate increases with radiation/leaf area but decreases with stress
photosynthetic_rate = (
    13
    + solar_radiation_mj * 0.40
    + leaf_area_cm2 * 0.025
    - drought_stress_index * 6.5
    - pathogen_pressure_index * 4.0
    - (avg_temp_c - 27) * 0.45
    + irrigation_frequency_per_week * 0.35
    + np.random.normal(0, 1.8, size=N)
)

photosynthetic_rate = np.clip(photosynthetic_rate, 1, 32)

# Stomatal conductance decreases under drought and heat
stomatal_conductance = (
    0.60
    - drought_stress_index * 0.38
    - pathogen_pressure_index * 0.10
    - (avg_temp_c - 27) * 0.02
    + irrigation_frequency_per_week * 0.02
    + np.random.normal(0, 0.05, size=N)
)

stomatal_conductance = np.clip(stomatal_conductance, 0.05, 0.9)

# Relative water content decreases strongly under drought and heat
relative_water_content_pct = (
    94
    + soil_moisture_pct * 0.25
    - drought_stress_index * 26
    - (avg_temp_c - 27) * 1.4
    + irrigation_frequency_per_week * 0.6
    + np.random.normal(0, 3.5, size=N)
)

relative_water_content_pct = np.clip(relative_water_content_pct, 25, 100)

# Fruit set rate increases during fruiting stage but declines with stress
fruit_set_rate_pct = (
    55 * stage_effect
    + fertilizer_rate_kg_ha * 0.08
    + irrigation_frequency_per_week * 1.5
    - drought_stress_index * 30
    - pathogen_pressure_index * 18
    - (avg_temp_c - 27) * 2.0
    + np.random.normal(0, 6, size=N)
)

fruit_set_rate_pct = np.clip(fruit_set_rate_pct, 0, 100)

# Stress class derived from drought + pathogen pressure
stress_score = drought_stress_index * 0.65 + pathogen_pressure_index * 0.35

stress_class = pd.cut(
    stress_score,
    bins=[0, 0.35, 0.65, 1.0],
    labels=["Low", "Moderate", "High"]
)

In [ ]:
# -----------------------------
# 4. Build Dataset
# -----------------------------

df = pd.DataFrame({
    "sample_id": np.arange(1, N+1),
    "location": location,
    "season": season,
    "avg_temp_c": avg_temp_c,
    "humidity_pct": humidity_pct,
    "rainfall_mm": rainfall_mm,
    "solar_radiation_mj": solar_radiation_mj,
    "soil_moisture_pct": soil_moisture_pct,
    "soil_ph": soil_ph,
    "altitude_m": altitude_m,
    "fertilizer_rate_kg_ha": fertilizer_rate_kg_ha,
    "irrigation_frequency_per_week": irrigation_frequency_per_week,
    "plant_age_weeks": plant_age_weeks,
    "leaf_area_cm2": leaf_area_cm2,
    "flowering_stage": flowering_stage,
    "drought_stress_index": drought_stress_index,
    "pathogen_pressure_index": pathogen_pressure_index,
    "chlorophyll_spad": chlorophyll_spad,
    "photosynthetic_rate": photosynthetic_rate,
    "stomatal_conductance": stomatal_conductance,
    "relative_water_content_pct": relative_water_content_pct,
    "fruit_set_rate_pct": fruit_set_rate_pct,
    "stress_class": stress_class
})

print(df.head())
print("\nDataset shape:", df.shape)

df.to_csv("synthetic_tomato_population.csv", index=False)

In [ ]:
# -----------------------------
# 5. EDA
# -----------------------------

plt.figure(figsize=(8,5))
df["chlorophyll_spad"].hist(bins=30)
plt.title("Distribution of Chlorophyll Content (SPAD)")
plt.xlabel("SPAD")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8,5))
df.boxplot(column="fruit_set_rate_pct", by="flowering_stage")
plt.title("Fruit Set Rate by Growth Stage")
plt.suptitle("")
plt.xlabel("Stage")
plt.ylabel("Fruit Set Rate (%)")
plt.show()

plt.figure(figsize=(8,5))
df.boxplot(column="relative_water_content_pct", by="season")
plt.title("Relative Water Content by Season")
plt.suptitle("")
plt.xlabel("Season")
plt.ylabel("RWC (%)")
plt.show()

In [ ]:
# -----------------------------
# 6. Multi-Output Predictive Modelling
# -----------------------------

target_cols = [
    "chlorophyll_spad",
    "photosynthetic_rate",
    "stomatal_conductance",
    "relative_water_content_pct",
    "fruit_set_rate_pct"
]

feature_cols = [c for c in df.columns if c not in target_cols + ["stress_class", "sample_id"]]

X = df[feature_cols]
y = df[target_cols]

cat_cols = ["location", "season", "flowering_stage"]
num_cols = [c for c in feature_cols if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=250, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print("\nModel Performance (Multi-output Regression):")
print(results_df)

In [ ]:
# -----------------------------
# 7. Scenario Simulation (Climate Stress)
# -----------------------------

scenario = X_test.copy()

scenario["avg_temp_c"] = scenario["avg_temp_c"] + 2
scenario["rainfall_mm"] = scenario["rainfall_mm"] * 0.7
scenario["drought_stress_index"] = np.clip(scenario["drought_stress_index"] + 0.2, 0, 1)

best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=250, random_state=42))
])

best_model.fit(X_train, y_train)

baseline_pred = best_model.predict(X_test)
scenario_pred = best_model.predict(scenario)

baseline_df = pd.DataFrame(baseline_pred, columns=target_cols)
scenario_df = pd.DataFrame(scenario_pred, columns=target_cols)

impact = scenario_df.mean() - baseline_df.mean()

print("\nScenario Impact: Climate Stress (+2°C, -30% rainfall, +0.2 drought)")
print(impact)

impact.plot(kind="bar", figsize=(10,5))
plt.title("Mean Change in Predicted Tomato Physiological Outcomes Under Climate Stress Scenario")
plt.ylabel("Change in Predicted Value")
plt.show()